## Agentic Pattern Used - Evaluator-Optimizer

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

In [ ]:
open_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
if open_api_key:
    print(f"the key starts with {open_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"the key starts with {google_api_key[:2]}")
else:
    print("gemini API Key not set")    

if groq_api_key:
   print(f"Groq API Key exists and begins {groq_api_key[:4]}")    
else:
    print("Groq API Key not set")

    
    

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]



In [ ]:
messages

In [ ]:
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages
)

question = response.choices[0].message.content
print (question)

In [ ]:
competitors = []
answers = []

messages = [{"role":"user", "content":question}]

In [ ]:
model_name = "gpt-5-nano"
response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "models/gemma-3n-e2b-it"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-20b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
#Check the models available for my api key
for model in gemini.models.list():
    print(model.id)

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
#Lets print all of them n see
print(competitors)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"


In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)



In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "models/gemma-3n-e2b-it"
response = gemini.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)

In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

It can fall under 2 patterns
1) Prompt Chaining
2) Evaluator-Optimizer(partially)

### I have attempted routing pattern in the below example. Depending on the current_query, the LLM would choose the appropriate format and call that function. Try to change the query from strings to int and see the change in output.

In [ ]:
system_prompt="""You are a helpful assistant that can perform mathematical calculations.
Respond  with exactly one of the following formats:
1. FUNCTION_CALL: function_name|input
2. FINAL_ANSWER: [number]

where the function_name can be one of the following:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time.
"""

current_query="""Calculate the sum of exponentials of word "INDIA"""

prompt = f"{system_prompt}\n\n Query: {current_query}"
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": prompt}])

print(response.choices[0].message.content)

In [ ]:
import math

def strings_to_chars_to_int(string):
    return [ord(char) for char in string]

def int_list_to_exponential_sum(int_list):
    int_list = eval(int_list)
    return sum(math.exp(x) for x in int_list)

def fibonacci_numbers(n):
    if n <= 0:
        return []
    fib_sequence = [0, 1]
    for _ in range(2, n):
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence[:n]

In [ ]:
response = response.choices[0].message.content
print(f"Response from model: {response}")

In [ ]:
_, function_info = response.split(":", 1)
_, function_info

In [ ]:
func_name, params = [x.strip() for x in function_info.split("|", 1)]

func_name, params

In [ ]:
def function_caller(func_name, params):
    """Simple function caller that maps function names to actual functions"""
    function_map = {
        "strings_to_chars_to_int": strings_to_chars_to_int,
        "int_list_to_exponential_sum": int_list_to_exponential_sum,
        "fibonacci_numbers": fibonacci_numbers
    }
    
    if func_name in function_map:
        return function_map[func_name](params)
    else:
        return f"Function {func_name} not found"

In [ ]:
iteration_result = function_caller(func_name, params)
print(f"Result of function call: {iteration_result}")

In [ ]:
system_prompt="""You are a helpful assistant that can perform mathematical calculations.
Respond  with exactly one of the following formats:
1. FUNCTION_CALL: function_name|input
2. FINAL_ANSWER: [number]

where the function_name can be one of the following:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time.
"""

current_query="""Calculate the fibonacci_numbers of 3"""

prompt = f"{system_prompt}\n\n Query: {current_query}"
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": prompt}])

print(response.choices[0].message.content)

In [ ]:
judge_llm = f"""You are a evaluator. Your job is to check the {system_prompt} follows the below rules
1. ✅ Structured Output Format  
   - Does the prompt enforce a predictable output format (e.g., FUNCTION_CALL, JSON, numbered steps)?  
   - Is the output easy to parse or validate?

2.✅ Instructional Framing  
   - Are there examples of desired behavior or “formats” to follow?  
   - Does the prompt define exactly how responses should look? 

2.✅ Confirm the response 
   - is the {response} in the desired output format?
   - Does the prompt define exactly how responses should look? 
 

Respond with a structured review in this format:
json

  "structured_output": true,
  "instructional_framing": true,
  "output_format_followed":true

"""


In [ ]:
result = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": judge_llm}])

print(result.choices[0].message.content)